<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Addressing and Routing

The earlier access exercises showed how to open a Duckiedrone shell. For a physical robot, that SSH connection depends on your base station finding a path across the network. If the connection stops working, repeating the SSH command tells you little about where the problem lies.

This notebook examines the configuration behind that path and introduces the concepts of network addressing and routing. You will identify a network interface and its addresses, interpret a subnet prefix, and read the routes Linux uses to send packets locally or through a gateway.

Use the Duckiedrone shell opened in [Notebook 13](./13-physical-duckiedrone-ssh-access.ipynb) or [Notebook 14](./14-virtual-duckiedrone-connections.ipynb), or follow the recorded physical-Duckiedrone example below (noting that virtual robots and Workspaces may show additional interfaces and different routes).

## Local networks

An SSH session to a physical Duckiedrone and a request to an internet website may leave through the same Wi-Fi interface, but follow different paths.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
Base station                 Duckiedrone
     |                            |
     +---- Wi-Fi access point ----+
                   |
                 Router
                   |
          Other networks / internet
  </pre>
  <figcaption>Figure 1: A common local network arrangement.</figcaption>
</figure>

A **wireless access point** connects Wi-Fi devices to the local network. A **switch** connects wired devices, while a **router** forwards traffic between networks. A home Wi-Fi router commonly combines all three roles.

Computers exchange data in **packets**, and Internet Protocol (**IP**) packets carry source and destination addresses. When  communication between the base station and Duckiedrone stays on the local network. Reaching an internet destination requires the router. A failed internet connection therefore does not, by itself, prevent a local SSH session.

## Interfaces and identifiers

A **network interface** is a computer’s connection to a network, such as Ethernet, Wi-Fi, or a virtual connection created by software.

Confirm your shell context with `hostname`, then inspect the interfaces:

```bash
ip -brief address
```

`-brief` requests compact output. These are selected rows from Amelia:

```text
lo               UNKNOWN        127.0.0.1/8 ::1/128
eth0             DOWN
wlan0            UP             192.168.1.201/24 metric 600
```

The Wi-Fi interface, `wlan0`, is up and has IPv4 address `192.168.1.201`. The Ethernet interface, `eth0`, is down and has no address shown.

`lo` is loopback: `127.0.0.1` and `::1` refer back to the current network environment. Its `UNKNOWN` state is normal here.

An `UP` interface does not prove another device is reachable. Interface names also vary: your Wi-Fi interface need not be called `wlan0`.

### Compare MAC and IP addresses

Inspect the interface’s link information:

```bash
ip -brief link
```

Amelia’s Wi-Fi row is:

```text
wlan0            UP             dc:a6:32:31:43:ad <BROADCAST,MULTICAST,UP,LOWER_UP>
```

The **Media Access Control (MAC) address** identifies the interface on its local connection. Compare its role with the other identifiers:

<table id="table-1" class="lx-table">
  <caption>Table 1: Identifiers for Amelia’s Wi-Fi interface.</caption>
  <thead>
    <tr><th>Identifier</th><th>Value</th><th>Purpose</th></tr>
  </thead>
  <tbody>
    <tr><td>Interface name</td><td><code>wlan0</code></td><td>Select this interface in Linux</td></tr>
    <tr><td>MAC address</td><td><code>dc:a6:32:31:43:ad</code></td><td>Identify it on the local link</td></tr>
    <tr><td>IPv4 address</td><td><code>192.168.1.201</code></td><td>Address IP packets to it</td></tr>
  </tbody>
</table>

For example, lab registration may require the Wi-Fi MAC address, while a program contacting the robot uses its hostname or IP address. A MAC address is not necessarily permanent: software and Wi-Fi privacy features can change it.

## Subnets and gateways

How does Amelia decide whether a destination is local?

Its address includes a **subnet prefix**:

```text
192.168.1.201/24
```

IPv4 addresses contain four decimal groups, each representing 8 bits. `/24` means the first 24 bits identify the network:

```text
192 . 168 . 1 . 201
└───────────┘   └─┘
 network part   host part
   24 bits       8 bits
```

Amelia’s subnet is therefore `192.168.1.0/24`. The equivalent **subnet mask** is `255.255.255.0`.

Suppose the base station shares that local network and has the illustrative address `192.168.1.42/24`:

| Destination from Amelia | Relationship to its Wi-Fi subnet |
| --- | --- |
| `192.168.1.42` | Inside: normally reached locally |
| `192.168.2.42` | Outside: requires an onward route |

Comparing the first three groups works because this prefix is `/24`; it is not a rule for every prefix length.

A **gateway** is a router used as the next step toward a destination. The **default gateway** handles destinations without a more specific route.

## Network names

You have already used names such as `DUCKIEDRONE_NAME.local` when connecting. Before sending packets, a program needs an address for that name.

**Name resolution** supplies the address; **routing** chooses where to send the packet:

```text
amelia.local → address lookup → 192.168.1.201 → route selection
```

This mapping is illustrative. Amelia’s interface capture alone does not verify that its name resolves.

DNS provides name-to-address mappings; `.local` names normally use multicast DNS (mDNS). [Notebook 21](./21-network-names-and-service-discovery.ipynb) develops these mechanisms and their connection to device discovery.

## Routing tables

Inspect the IPv4 routes:

```bash
ip route
```

These are selected entries from Amelia’s recorded table:

```text
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
192.168.1.1 dev wlan0 proto dhcp scope link src 192.168.1.201 metric 600
```

### Read the local route

For the example base station at `192.168.1.42`, the matching subnet route is:

```text
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
```

| Field | Meaning |
| --- | --- |
| `192.168.1.0/24` | Destination subnet |
| `dev wlan0` | Outgoing interface |
| `scope link` | Destinations are directly connected |
| `src 192.168.1.201` | Preferred source address for traffic Amelia originates |

There is no `via` gateway. Traffic still passes through the Wi-Fi access point, but does not need routing into another network.

### Read the default route

For the illustrative destination `192.168.2.42`, no specific entry matches. Amelia uses:

```text
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
```

The first hop is gateway `192.168.1.1`, reached through `wlan0`:

```text
Amelia → 192.168.1.1 → ? → 192.168.2.42
```

The question mark matters: Amelia’s table does not establish whether the gateway can forward the packet successfully.

### Understand which entry wins

For ordinary routing in this table, the **most specific matching destination prefix** wins—not the first printed line.

| Destination | Selected entry |
| --- | --- |
| `192.168.1.42` | Local subnet `192.168.1.0/24` |
| `192.168.2.42` | `default` |
| `192.168.1.1` | Route specifically to `192.168.1.1` |

`proto kernel` identifies a kernel-installed route. `proto dhcp` identifies one supplied through automatic configuration using Dynamic Host Configuration Protocol (DHCP). A `metric` helps choose between otherwise comparable routes; lower values are preferred. See the [`ip route` manual](https://man7.org/linux/man-pages/man8/ip-route.8.html).

**Question:** If the default route disappeared but the Wi-Fi subnet route remained, would Amelia still have a route to `192.168.1.42`?

<details>
<summary>Reveal answer</summary>

Yes. The local `192.168.1.0/24` entry still covers that address. Destinations that depended on the default route would lose their route.

</details>

### Try it

Using your own output, identify:

1. A non-loopback interface with an IP address and prefix.
2. The route for a directly connected subnet.
3. The default gateway, if one is configured.

Compare these with Amelia’s example. Virtual environments may have several software-created interfaces; their names and addresses need not match.

These commands inspect configuration, not successful delivery. [Notebook 22](./22-network-diagnostics-and-testing.ipynb) adds connection tests.

## Address families together

Amelia’s capture also contains `::1`, an **IPv6** address. IPv6 uses 128-bit addresses, compared with IPv4’s 32 bits.

IPv6 addresses use hexadecimal groups separated by colons. One consecutive run of zero groups can be shortened to `::`:

```text
2001:db8:0:0:0:0:0:42 → 2001:db8::42
```

This address is reserved for documentation. Two forms to recognize in real output are:

| Address | Meaning |
| --- | --- |
| `::1` | IPv6 loopback |
| An address beginning `fe80::` | A common link-local form; routers do not forward it beyond the local link |

These forms are defined in the [IPv6 addressing specification](https://www.rfc-editor.org/rfc/rfc4291.html).

A computer can use IPv4 and IPv6 together, called **dual stack**, but success with one does not prove the other works. Amelia’s capture shows IPv6 loopback and no IPv6 address on `wlan0`. Likewise, a link-local IPv6 address alone does not establish IPv6 internet access.

## Further reading

The Linux [`ip address` manual](https://man7.org/linux/man-pages/man8/ip-address.8.html) describes assigned addresses and interface information.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
